# 02 Calibration, land masking, multilooking and polarimetric products

This notebook takes one Biomass Level-1a SCS frame and turns it into the products every later notebook works from (report Section 4.1):

1. read the four complex channels, checking the band order instead of assuming it
2. geolocate every pixel with range-Doppler geocoding on a 20 x 20 tie point grid
3. mask land with ESA WorldCover and push the mask a few km out from the coast
4. multilook into cells that are roughly square on the ground
5. convert to sigma nought and plot HH, VV, cross-pol and the Pauli composite
6. copolar phase difference (CPD) and HH-VV coherence, using circular statistics
7. entropy and alpha, plus three instrument checks (Faraday rotation, HV vs VH, CPD and HH/VV against incidence angle)

Everything gets saved to one `.npz` file per frame, which notebooks 04 and 06 read. The last section stacks the frames of one pass into a strip.

Only the first code cell needs changing for a new frame.

In [ ]:
# ---- the only cell that needs editing ----
from pathlib import Path

PRODUCT = Path(r"C:\Users\Orlaith.Doyle\BIO_S1_SCS__1S_20260630T181207_20260630T181228_C_G01_M01_C01_T033_F156_01_DAB2C1")  # unzipped frame folder
OUT = Path(r"C:\Users\Orlaith.Doyle\biomass_processed")

CELL_M = 250            # rough size of a multilooked cell on the ground (m)
USE_WORLDCOVER = True   # False for frames far out at sea, saves the download
COAST_BUFFER_KM = 5     # land mask pushed this far out to sea, 3 to 8 km depending on the coast
MASK_STEP = 8           # land mask is sampled every 8 pixels then filled back in, plenty at 10 m worldcover

# only used for the pure bragg line in the instrument check
SSS_GUESS, SST_GUESS, WIND_GUESS = 35.0, 25.0, 7.0

CPD_CMAP = "gist_rainbow"   # tried loads of wrapping cmaps, this one shows the phase structure best

In [ ]:
import sys
sys.path.append(str(Path.cwd()))      # bm_tools.py lives next to the notebooks
import numpy as np
import matplotlib.pyplot as plt
import bm_tools as bm

OUT.mkdir(parents=True, exist_ok=True)
meta = bm.parse_name(PRODUCT.name)
TAG = f"T{meta['track']:03d}_F{meta['frame']:03d}_{meta['start']:%Y%m%dT%H%M}" if meta["start"] else PRODUCT.name[:40]
print(TAG, meta["swath"])


def save(fig, what):
    """every figure goes to disk before it is shown"""
    p = OUT / f"{TAG}_{what}.png"
    fig.savefig(p, dpi=200, bbox_inches="tight")
    print("saved", p.name)

## 1. Read the channels

Each frame has one amplitude TIFF and one phase TIFF, both with four bands. The SCS TIFFs store the bands as VV, VH, HV, HH, which is the reverse of the order listed in the annotation XML. For a long time I read them in the annotation order, which swapped HH and VV. That flips the sign of the CPD and turns HH/VV upside down, so it quietly broke every phase result before it was found.

So the order is now read from the band metadata in the TIFF itself, and if it can't be found it falls back to VV, VH, HV, HH with a loud warning. The cell also prints the annotation order next to it so the two can be compared by eye.

In [ ]:
slc, order = bm.read_slc(PRODUCT)
ann = bm.read_annotation(PRODUCT)
shape = slc["HH"].shape

print("band order in the tiff :", order)
print("order in the annotation:", ann["pols_xml"])
print("orbit pass:", ann["orbit_pass"], "  shape:", shape, "  f0 = %.1f MHz" % (ann["f0"] / 1e6))

# quick sanity check: over the sea VV should be brighter than HH on average.
# if it's the other way round the band order is probably still wrong
r = np.nanmean(np.abs(slc["HH"][::7, ::7]) ** 2) / np.nanmean(np.abs(slc["VV"][::7, ::7]) ** 2)
print("frame mean HH/VV = %.2f dB" % (10 * np.log10(r)), "<- should be negative over open ocean")

## 2. Geolocation

The satellite position comes from the orbit state vectors (cubic spline), and each pixel goes where its slant range sphere meets the zero-Doppler plane on the WGS84 ellipsoid. Doing that for every pixel is slow, so it's solved exactly on a 20 x 20 grid of tie points and interpolated in between.

The first version just interpolated between the four footprint corners and was up to 55 km out in the middle of the frame. The check at the bottom geocodes a few random pixels exactly and compares them with the interpolated grid.

The incidence angle comes out of the same calculation and is what the sigma nought conversion uses later.

In [ ]:
rows_t, cols_t, lat_t, lon_t, inc_t = bm.geocode_grid(ann, shape, n=20, look=ann["look"])

d_az, d_rg = bm.pixel_spacing(rows_t, cols_t, lat_t, lon_t)
heading = bm.track_heading(lat_t, lon_t)
print("ground spacing: %.2f m azimuth, %.2f m range" % (d_az, d_rg))
print("incidence %.1f to %.1f deg, heading %.1f deg, %s looking" % (inc_t.min(), inc_t.max(), heading, ann["look"]))
print("corners (lat, lon):")
for i, j in [(0, 0), (0, -1), (-1, 0), (-1, -1)]:
    print("   %.4f, %.4f" % (lat_t[i, j], lon_t[i, j]))

# check the tie grid against exact geocoding at a few random pixels
rng = np.random.default_rng(1)
err = []
for _ in range(6):
    rr, cc = rng.uniform(0, shape[0] - 1), rng.uniform(0, shape[1] - 1)
    la, lo, _ = bm.geocode_point(ann, rr, cc, guess=[np.mean(lat_t), np.mean(lon_t)], look=ann["look"])
    la_i = bm.grid_interp(rows_t, cols_t, lat_t, [rr], [cc])[0, 0]
    lo_i = bm.grid_interp(rows_t, cols_t, lon_t, [rr], [cc])[0, 0]
    err.append(bm.haversine_m(la, lo, la_i, lo_i))
print("tie grid vs exact: max %.2f m" % max(err))

## 3. Land masking

Land and coastal structures are very bright at P-band and would swamp any ocean average, so they have to go before anything is averaged.

The mask comes from ESA WorldCover (10 m, 2021) sampled at the geocoded pixel positions. Class 80 is permanent water and class 0 is "no data", which is what WorldCover has offshore, so both count as water. Tiles that don't exist at all are open ocean. The mask is then grown out to sea by `COAST_BUFFER_KM` so the bright coastline and the shallow, messy water right next to it are left out too.

It's sampled every `MASK_STEP` pixels and filled back in, which is more than fine for a buffer that is kilometres wide.

In [ ]:
r_m = np.arange(0, shape[0], MASK_STEP)
c_m = np.arange(0, shape[1], MASK_STEP)
lat_m = bm.grid_interp(rows_t, cols_t, lat_t, r_m, c_m)
lon_m = bm.grid_interp(rows_t, cols_t, lon_t, r_m, c_m)

if USE_WORLDCOVER:
    print("sampling worldcover at", lat_m.size, "points...")
    water_m = bm.worldcover_water(lat_m, lon_m)
else:
    water_m = np.ones(lat_m.shape, bool)

land_m = ~water_m
land_m |= bm.near_land(land_m, d_az * MASK_STEP, d_rg * MASK_STEP, COAST_BUFFER_KM)

# back to full resolution
water = np.repeat(np.repeat(~land_m, MASK_STEP, 0), MASK_STEP, 1)[:shape[0], :shape[1]]
print("water fraction of the frame: %.1f %%" % (100 * water.mean()))

fig, ax = plt.subplots(figsize=(5, 6))
ax.pcolormesh(lon_m, lat_m, land_m, cmap="Greys", shading="auto")
ax.set_title(f"{TAG}\nland + {COAST_BUFFER_KM} km buffer (black)")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
save(fig, "landmask")
plt.show()

## 4. Multilooking

A single pixel is too noisy to use because of speckle, so neighbouring pixels are averaged. The window is set so each averaged cell is about `CELL_M` on the ground in both directions, which means the ratio of azimuth to range pixels follows the ratio of the ground spacings (report Eq. 22):

N_az : N_rg = dx_rg / dx_az : 1

Land pixels get zero weight in the average, so they don't leak into the ocean cells, and any cell that is less than half water is dropped.

All the second order products are made in one go here (channel powers, the HH-VV cross product, the coherency matrix terms and the Bickel and Bates term for Faraday rotation), so every later step works from the same averages. This is the slow cell.

In [ ]:
NAZ = max(1, int(round(CELL_M / d_az)))
NRG = max(1, int(round(CELL_M / d_rg)))
print(f"window {NAZ} az x {NRG} rg = {NAZ * NRG} pixels, cell about {NAZ * d_az:.0f} x {NRG * d_rg:.0f} m")

ml = bm.ml_stats(slc, water, NAZ, NRG)

# geometry at the centre of each multilooked cell
nr, nc = ml["hh"].shape
r_c = np.arange(nr) * NAZ + (NAZ - 1) / 2
c_c = np.arange(nc) * NRG + (NRG - 1) / 2
LAT = bm.grid_interp(rows_t, cols_t, lat_t, r_c, c_c)
LON = bm.grid_interp(rows_t, cols_t, lon_t, r_c, c_c)
INC = bm.grid_interp(rows_t, cols_t, inc_t, r_c, c_c)

# pixels are correlated so the real number of looks is lower than NAZ * NRG. rough estimate
# from how much VV varies between neighbouring cells (a few 3x3 blocks of flat sea)
v = ml["vv"][: nr // 3 * 3, : nc // 3 * 3].reshape(nr // 3, 3, nc // 3, 3)
enl = np.nanmedian(np.nanmean(v, (1, 3)) ** 2 / np.nanvar(v, (1, 3)))
print("estimated ENL about %.0f (nominal %d)" % (enl, NAZ * NRG))

## 5. Backscatter

The Level-1a pixel values are beta nought, brightness per unit area in the slant range plane. Multiplying by sin of the local incidence angle gives sigma nought, per unit area on the ground (report Eq. 23):

sigma0_pq = <|S_pq|^2> sin(theta)

I checked this against the sigmaNought table that comes with each product and they agree to rounding. The cross-pol channel is the mean of HV and VH.

The Pauli composite uses |HH - VV|^2 for red (double bounce), the cross-pol for green (volume) and |HH + VV|^2 for blue (surface). Clean sea should come out almost entirely blue.

The CPD and coherence don't depend on this step, since the sin(theta) factor is the same for HH and VV and cancels.

In [ ]:
sinI = np.sin(np.radians(INC))
s0 = {"HH": ml["hh"] * sinI, "VV": ml["vv"] * sinI, "X": ml["x"] * sinI}
s0db = {k: 10 * np.log10(v) for k, v in s0.items()}

fig, axs = plt.subplots(1, 4, figsize=(18, 6), sharey=True, layout="constrained")
for a, k in zip(axs[:3], ["HH", "VV", "X"]):
    lo_, hi_ = np.nanpercentile(s0db[k], [2, 98])
    im = a.pcolormesh(LON, LAT, s0db[k], cmap="gray", vmin=lo_, vmax=hi_, shading="auto")
    plt.colorbar(im, ax=a, shrink=0.7, label="dB")
    a.set_title({"X": "cross-pol (HV+VH)/2"}.get(k, k) + " sigma0")

def stretch(x):
    x = np.sqrt(x)
    lo_, hi_ = np.nanpercentile(x, [2, 98])
    return np.clip((x - lo_) / (hi_ - lo_), 0, 1)

rgb = np.dstack([stretch(ml["t22"]), stretch(ml["t33"]), stretch(ml["t11"])])
rgb[~np.isfinite(rgb)] = 1
try:
    axs[3].pcolormesh(LON, LAT, rgb, shading="auto")      # rgb pcolormesh needs matplotlib 3.8 or newer
except Exception:
    axs[3].imshow(rgb, aspect="auto", origin="upper")      # older matplotlib, radar geometry instead
    axs[3].set_xlabel("range px")
axs[3].set_title("Pauli (R dbl, G vol, B surf)")
for a in axs:
    a.set_xlabel("lon")
axs[0].set_ylabel("lat")
fig.suptitle(TAG)
save(fig, "sigma0_pauli")
plt.show()

## 6. CPD and coherence

The complex HH-VV correlation is

gamma = <S_HH S_VV*> / sqrt(<|S_HH|^2> <|S_VV|^2>)

Its size is the coherence and its angle is the CPD. Because the CPD is an angle it wraps at +-180, so every average and spread here is done on the unit circle, not on the numbers.

For display each frame gets its own colour range, its circular mean +-1 circular standard deviation. Most of the useful data sits inside that, and it stops the whole map being one colour when a frame's mean is a bit off. To keep it honest across the wrap, what's plotted is the deviation from the frame mean, and the colour bar is labelled with the real phase.

In [ ]:
COH = np.abs(ml["hhvv"]) / np.sqrt(ml["hh"] * ml["vv"])
CPD = np.angle(ml["hhvv"])                          # radians

ok = np.isfinite(CPD)
mu, sd = bm.circ_mean(CPD[ok]), bm.circ_std(CPD[ok])
dev = np.degrees(np.angle(np.exp(1j * (CPD - mu))))   # deviation from the frame mean, wrap safe
print("frame CPD = %.2f +- %.2f deg (circular), coherence median %.3f"
      % (np.degrees(mu), np.degrees(sd), np.nanmedian(COH)))

fig, axs = plt.subplots(1, 3, figsize=(16, 6), layout="constrained")
im = axs[0].pcolormesh(LON, LAT, dev, cmap=CPD_CMAP, vmin=-np.degrees(sd), vmax=np.degrees(sd), shading="auto")
cb = plt.colorbar(im, ax=axs[0], shrink=0.7)
ticks = np.linspace(-np.degrees(sd), np.degrees(sd), 5)
cb.set_ticks(ticks); cb.set_ticklabels([f"{np.degrees(mu) + t:.1f}" for t in ticks]); cb.set_label("CPD (deg)")
axs[0].set_title("CPD, mean +- 1 circ. std")

im = axs[1].pcolormesh(LON, LAT, COH, cmap="viridis", vmin=0.8, vmax=1, shading="auto")
plt.colorbar(im, ax=axs[1], shrink=0.7, label="|gamma|")
axs[1].set_title("HH-VV coherence")

m_d, s_d = np.degrees(mu), np.degrees(sd)
axs[2].hist(m_d + dev[ok], bins=80, range=(m_d - 5 * s_d, m_d + 5 * s_d), color="0.4")   # unwrapped around the mean
for x in (np.degrees(mu) - np.degrees(sd), np.degrees(mu), np.degrees(mu) + np.degrees(sd)):
    axs[2].axvline(x, color="r", lw=1)
axs[2].set_xlabel("CPD (deg)"); axs[2].set_title("histogram, red = mean +- 1 sd")
for a in axs[:2]:
    a.set_xlabel("lon"); a.set_ylabel("lat")
fig.suptitle(TAG)
save(fig, "cpd_coherence")
plt.show()

# precision to expect per cell from report eq 6, with the estimated ENL
print("expected CPD precision per cell: %.1f deg" % bm.cpd_std_deg(enl, np.nanmedian(COH)))

## 7. Entropy and alpha

The eigenvalues of the 3 x 3 coherency matrix split each cell's echo into three independent mechanisms. The entropy H says how mixed they are (0 is one mechanism, 1 is fully random) and the mean alpha says what kind dominates (about 0 deg surface, 45 deg volume, 90 deg double bounce).

Clean sea is a single surface mechanism so it sits at low H and low alpha. Rain, slicks, ships and fronts push cells up and to the right, which is what notebook 06 uses.

In [ ]:
H, ALPHA = bm.h_alpha(ml)

fig, axs = plt.subplots(1, 3, figsize=(16, 5.5), layout="constrained")
im = axs[0].pcolormesh(LON, LAT, H, cmap="magma", vmin=0, vmax=1, shading="auto")
plt.colorbar(im, ax=axs[0], shrink=0.7); axs[0].set_title("entropy H")
im = axs[1].pcolormesh(LON, LAT, ALPHA, cmap="magma", vmin=0, vmax=60, shading="auto")
plt.colorbar(im, ax=axs[1], shrink=0.7, label="deg"); axs[1].set_title("mean alpha")
g = np.isfinite(H)
axs[2].hist2d(H[g], ALPHA[g], bins=[100, 90], range=[[0, 1], [0, 90]], cmap="Greys", cmin=1)
axs[2].set_xlabel("H"); axs[2].set_ylabel("alpha (deg)"); axs[2].set_title("H-alpha plane")
for a in axs[:2]:
    a.set_xlabel("lon"); a.set_ylabel("lat")
fig.suptitle(TAG)
save(fig, "h_alpha")
plt.show()

## 8. Instrument checks

Three things are checked before reading anything geophysical into a frame:

- **Faraday rotation.** The Level-1 processor estimates it and removes it. If the product carries its estimate it's printed. The residual left in the data is estimated with the Bickel and Bates method, Omega = 1/4 arg<Z12 Z21*>. Over the sea the cross-pol is weak so per cell values are noisy and only the frame mean means anything.
- **HV vs VH.** By reciprocity these should be the same, so their phase difference should sit around zero.
- **HH/VV and CPD against incidence angle,** binned across the swath, with the pure Bragg prediction from the guessed salinity, temperature and wind in the first cell. The CPD rises from near to far range by a few degrees on every swath, far more than any salinity effect, which is why notebook 04 removes it.

In [ ]:
# faraday rotation: whatever the product says, then what's left in the data
try:
    root = bm._xml_root(PRODUCT)
    fr = root.xpath("//*[contains(local-name(), 'araday')]")
    vals = [float(e.text) for e in fr if e.text and e.text.strip().replace(".", "", 1).lstrip("-").isdigit()]
    print("faraday rotation in the annotation:", f"{np.mean(vals):.3f}" if vals else "not found")
except Exception as e:
    print("couldn't read faraday from the annotation:", e)

OMEGA = 0.25 * np.angle(ml["z1221"])
print("residual faraday (bickel and bates), frame circular mean: %.2f deg"
      % np.degrees(bm.circ_mean(4 * OMEGA[np.isfinite(OMEGA)]) / 4))

HVVH = np.angle(ml["hvvh"])
print("HV - VH phase: mean %.2f deg, circ std %.1f deg"
      % (np.degrees(bm.circ_mean(HVVH[np.isfinite(HVVH)])), np.degrees(bm.circ_std(HVVH[np.isfinite(HVVH)]))))

# against incidence
bins = np.linspace(np.nanmin(INC), np.nanmax(INC), 16)
mid = (bins[:-1] + bins[1:]) / 2
ratio_b, cpd_b = [], []
for a, b in zip(bins[:-1], bins[1:]):
    s = (INC >= a) & (INC < b) & np.isfinite(CPD)
    ratio_b.append(10 * np.log10(np.nansum(ml["hh"][s]) / np.nansum(ml["vv"][s])) if s.any() else np.nan)
    cpd_b.append(np.degrees(np.angle(np.nansum(ml["hhvv"][s]))) if s.any() else np.nan)
ratio_b, cpd_b = np.array(ratio_b), np.array(cpd_b)

eps = bm.eps_klein_swift(SSS_GUESS, SST_GUESS, ann["f0"])
th = np.linspace(mid.min(), mid.max(), 50)
hh_m, vv_m, cpd_m = bm.pure_bragg(eps, th, ann["f0"], WIND_GUESS)

g = np.isfinite(cpd_b)
slope = np.polyfit(mid[g], cpd_b[g], 1)[0]
print("CPD change across the swath: %.2f deg" % (slope * (mid.max() - mid.min())))

fig, axs = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
axs[0].plot(mid, ratio_b, "o-", label="Biomass")
axs[0].plot(th, 10 * np.log10(hh_m / vv_m), "k--", label="pure Bragg")
axs[0].set_ylabel("HH/VV (dB)")
axs[1].plot(mid, cpd_b, "o-", label="Biomass")
axs[1].plot(th, cpd_m, "k--", label="pure Bragg")
axs[1].set_ylabel("CPD (deg)")
for a in axs[:2]:
    a.set_xlabel("incidence (deg)"); a.legend()
im = axs[2].pcolormesh(LON, LAT, np.degrees(HVVH), cmap="twilight", vmin=-180, vmax=180, shading="auto")
plt.colorbar(im, ax=axs[2], shrink=0.8, label="deg"); axs[2].set_title("HV - VH phase")
fig.suptitle(f"{TAG}  (model: S={SSS_GUESS} psu, T={SST_GUESS} C, U10={WIND_GUESS} m/s)")
save(fig, "instrument_checks")
plt.show()

## 9. Save

Everything later notebooks need goes into one compressed `.npz` per frame: the multilooked second order products (so cells can be averaged again properly further down the line instead of averaging dB or angles), the derived maps, the geometry and the metadata.

In [ ]:
np.savez_compressed(
    OUT / f"{TAG}.npz",
    product=PRODUCT.name, tag=TAG, swath=meta["swath"], track=meta["track"], frame=meta["frame"],
    t0=str(ann["t0"]), orbit_pass=ann["orbit_pass"], f0=ann["f0"], naz=NAZ, nrg=NRG,
    d_az=d_az, d_rg=d_rg, heading=heading, enl=enl,
    lat=LAT, lon=LON, inc=INC, wfrac=ml["wfrac"],
    hh=ml["hh"], vv=ml["vv"], x=ml["x"], hhvv=ml["hhvv"], hvvh=ml["hvvh"],
    t11=ml["t11"], t22=ml["t22"], t33=ml["t33"],
    s0hh=s0["HH"], s0vv=s0["VV"], s0x=s0["X"], coh=COH, cpd=CPD, H=H, alpha=ALPHA, omega=OMEGA)
print("saved", OUT / f"{TAG}.npz")

## 10. Stack the frames of one pass into a strip

Once every frame of a pass has been through the cells above, this puts them together. Frames from the same track and date are drawn on one map, each with its own colour range (its own circular mean +-1 sd), so the spatial pattern stays visible even when neighbouring frames have slightly different mean phases. That frame to frame offset is instrument and geometry, not sea, so it's not worth keeping on the plot.

In [ ]:
STRIP_TRACK = meta["track"]
STRIP_DATE = f"{meta['start']:%Y%m%d}" if meta["start"] else ""

files = sorted(OUT.glob(f"T{STRIP_TRACK:03d}_F*_{STRIP_DATE}T*.npz"))
print(len(files), "frames for this pass")

fig, axs = plt.subplots(1, 2, figsize=(11, 10), sharey=True)
for f in files:
    d = np.load(f)
    cpd = d["cpd"]; g = np.isfinite(cpd)
    if g.sum() < 50:
        continue
    m_, s_ = bm.circ_mean(cpd[g]), bm.circ_std(cpd[g])
    axs[0].pcolormesh(d["lon"], d["lat"], np.angle(np.exp(1j * (cpd - m_))) / s_,
                      cmap=CPD_CMAP, vmin=-1, vmax=1, shading="auto")
    axs[1].pcolormesh(d["lon"], d["lat"], 10 * np.log10(d["s0vv"]), cmap="gray",
                      vmin=np.nanpercentile(10 * np.log10(d["s0vv"]), 2),
                      vmax=np.nanpercentile(10 * np.log10(d["s0vv"]), 98), shading="auto")
    axs[0].text(np.nanmean(d["lon"]), np.nanmean(d["lat"]), f"F{int(d['frame']):03d}", fontsize=7)
axs[0].set_title("CPD, each frame its own mean +- 1 sd")
axs[1].set_title("VV sigma0 (dB, per frame stretch)")
for a in axs:
    a.set_xlabel("lon")
axs[0].set_ylabel("lat")
fig.suptitle(f"T{STRIP_TRACK:03d} {STRIP_DATE}")
save(fig, f"strip_T{STRIP_TRACK:03d}_{STRIP_DATE}")
plt.show()